In [ ]:
import pandas as pd

X = pd.read_csv('/content/data/train.csv')

y = X['price_range']
X.drop(['price_range'], axis=1, inplace=True)

from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2)

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

gb_params = {
    'learning_rate' : [0.01, 0.2, 0.3],
    'n_estimators' : [100, 200, 300],
    'max_depth' : [3, 5, 6],
    'subsample' : [0.7, 0.9, 1.0],
    'colsample_bytree' : [0.7, 0.9, 1.0],
    'gamma' : [0.1, 0.2, 0.3],
}

model = XGBClassifier()

grid_search = GridSearchCV(estimator=model,
                           param_grid=gb_params,
                           cv=5,
                           scoring='accuracy',
                           n_jobs=-1,
                           verbose=1)

grid_search.fit(X_train, y_train)

In [ ]:
import joblib
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

model = grid_search.best_estimator_

joblib.dump(model, 'XGBClassifier.pkl')

y_preds = model.predict(X_valid)
accuracy_score(y_valid, y_preds)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_sc = scaler.fit_transform(X_train)

X_scaled = pd.DataFrame(X_sc)
X_scaled.columns = X_train.columns
X_valid_scaled = pd.DataFrame(scaler.transform(X_valid))
X_valid_scaled.columns = X_valid.columns

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

lr_params ={
    'penalty': ['l1', 'l2', 'elasticnet', 'none'],
    'C': [0.1, 1, 10, 100],
    'solver': ['liblinear', 'saga'],
    'max_iter': [100, 200, 300],
    'l1_ratio': [0.5],
    'class_weight': [None, 'balanced']
}

model = LogisticRegression()

grid_search = GridSearchCV(model, lr_params, cv=5,
                           scoring='accuracy', n_jobs=-1,
                           verbose=2)

grid_search.fit(X_scaled, y_train)

In [ ]:
from sklearn.metrics import accuracy_score
model = grid_search.best_estimator_

preds = model.predict(X_valid_scaled)
print(classification_report(y_valid, preds))

In [ ]:
import joblib

joblib.dump(model, 'LogisticRegression.pkl')